# שלב 03 — ניתוח תיאורי של הרשת

לפני שמחפשים 'מי התחנה הכי חשובה', צריך להבין איך הרשת בנויה: כמה רכיבים קשורים יש, האם היא מחוברת, ומהן נקודות התורפה המבניות שלה (Articulation Points ו‑Bridges).

In [ ]:
# התקנת הספריות הנדרשות (להריץ פעם אחת; אפשר לדלג אם כבר מותקנות)
%pip install pandas numpy networkx matplotlib seaborn python-bidi

In [ ]:
from pathlib import Path
import pickle, json
import pandas as pd
import numpy as np
import networkx as nx


def find_repo_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / "israel-public-transportation").exists():
            return cand
    raise FileNotFoundError("repo root not found - set ROOT manually")


ROOT = find_repo_root(Path.cwd())
BASE = ROOT / "public_transport_network_notebooks"
GRAPH_DIR = BASE / "outputs" / "02_graph_construction"
OUT_DIR = BASE / "outputs" / "03_network_descriptive_analysis"
FIG_DIR = BASE / "figures" / "03_network_descriptive_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("GRAPH_DIR:", GRAPH_DIR)

In [ ]:
import re
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from bidi.algorithm import get_display


def _fix(t):
    """מסדר טקסט עברי לתצוגה נכונה (bidi). אנגלית ומספרים נשארים כמו שהם."""
    if isinstance(t, str) and any(0x590 <= ord(c) <= 0x5FF for c in t):
        return get_display(t)
    return t


import matplotlib.text as _mt
if not getattr(_mt.Text, "_bidi", False):
    _orig = _mt.Text.set_text
    def _set(self, s):
        if isinstance(s, str) and getattr(self, "_disp", None) == s:
            return _orig(self, s)
        f = _fix(s)
        if isinstance(f, str):
            self._disp = f
        return _orig(self, f)
    _mt.Text.set_text = _set
    _mt.Text._bidi = True

sns.set_theme(style="whitegrid", font_scale=1.1)
matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
print("עברית בגרפים מופעלת")

## טעינת הגרפים

טוענים את הגרף המכוון והלא מכוון שנבנו בשלב 02.

In [ ]:
def load_graphs():
    with open(GRAPH_DIR / "graph_undirected.pkl", "rb") as f:
        G = pickle.load(f)
    with open(GRAPH_DIR / "graph_directed.pkl", "rb") as f:
        D = pickle.load(f)
    return G, D


G, D = load_graphs()
print(f"גרף נטען: {G.number_of_nodes():,} צמתים, {G.number_of_edges():,} קשתות")

## סטטיסטיקות מבניות

מחשבים: רכיבים קשורים, גודל הרכיב הגדול, רכיבים חזקים/חלשים (בגרף המכוון), והכי חשוב — **Articulation Points** (תחנות שהסרתן מפצלת את הרשת) ו‑**Bridges** (מקטעים שהסרתם מפצלת). החישוב של AP ו‑Bridges עשוי לקחת כדקה.

In [ ]:
def compute_summary(G, D):
    components = sorted(nx.connected_components(G), key=len, reverse=True)
    largest = len(components[0])
    ap = list(nx.articulation_points(G))
    br = list(nx.bridges(G))

    wcc = list(nx.weakly_connected_components(D))
    scc = list(nx.strongly_connected_components(D))

    degrees = [d for _, d in G.degree()]
    return {
        "num_nodes": G.number_of_nodes(),
        "num_edges_undirected": G.number_of_edges(),
        "num_edges_directed": D.number_of_edges(),
        "density": round(nx.density(G), 6),
        "avg_degree": round(np.mean(degrees), 2),
        "max_degree": int(np.max(degrees)),
        "min_degree": int(np.min(degrees)),
        "num_connected_components": len(components),
        "largest_component_nodes": largest,
        "largest_component_share": round(largest / G.number_of_nodes(), 4),
        "num_weakly_connected": len(wcc),
        "num_strongly_connected": len(scc),
        "largest_scc_nodes": max(len(c) for c in scc),
        "num_articulation_points": len(ap),
        "num_bridges": len(br),
    }, ap, br


print("מחשב סטטיסטיקות (AP ו-Bridges עשויים לקחת כדקה) ...")
summary, ap, br = compute_summary(G, D)
for k, v in summary.items():
    print(f"  {k}: {v}")

## שמירת הסיכום ונקודות התורפה

שומרים את הסיכום ל‑JSON/CSV ואת רשימות ה‑Articulation Points וה‑Bridges.

In [ ]:
with open(OUT_DIR / "network_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
pd.DataFrame([summary]).to_csv(OUT_DIR / "network_summary.csv", index=False, encoding="utf-8-sig")

nodes_df = pd.DataFrame([{"stop_id": n, **G.nodes[n]} for n in G.nodes()])
ap_set = set(ap)
ap_df = nodes_df[nodes_df["stop_id"].isin(ap_set)].copy()
ap_df.to_csv(OUT_DIR / "articulation_points.csv", index=False, encoding="utf-8-sig")

br_rows = [{"from_stop": u, "to_stop": v, "trip_frequency": int(G[u][v].get("weight", 1))} for u, v in br]
pd.DataFrame(br_rows).to_csv(OUT_DIR / "bridges.csv", index=False, encoding="utf-8-sig")
print(f"  {len(ap_df)} Articulation Points, {len(br_rows)} Bridges")

## גרף: התפלגות דרגות

ההיסטוגרמה והתצוגה ב‑log-log בודקות אם הרשת היא 'Scale-Free': מעט תחנות עם דרגה גבוהה (hubs) והרבה תחנות עם דרגה נמוכה.

In [ ]:
def plot_degree_distribution(G, fig_dir):
    degrees = [d for _, d in G.degree()]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].hist(degrees, bins=60, color="#2563eb", edgecolor="white", linewidth=0.3)
    axes[0].set_xlabel("Degree (מספר שכנים)")
    axes[0].set_ylabel("מספר תחנות")
    axes[0].set_title("התפלגות דרגות — רשת תחבורה ישראל")

    deg_counts = pd.Series(degrees).value_counts().sort_index()
    deg_counts = deg_counts[deg_counts.index > 0]
    axes[1].scatter(np.log10(deg_counts.index), np.log10(deg_counts.values), s=8, color="#dc2626", alpha=0.6)
    axes[1].set_xlabel("log10(Degree)")
    axes[1].set_ylabel("log10(Count)")
    axes[1].set_title("Log-Log Degree Distribution (בדיקת Power Law)")

    plt.tight_layout()
    plt.savefig(fig_dir / "degree_distribution.png", dpi=150)
    plt.show()


plot_degree_distribution(G, FIG_DIR)

## גרף: מפת הרשת הארצית

כל תחנה כנקודה, הצבע והגודל לפי הדרגה. רואים מיד את הריכוז במרכז הארץ.

In [ ]:
def plot_network_map(G, fig_dir):
    nwc = [(n, d) for n, d in G.nodes(data=True) if d.get("lat") and d.get("lon")]
    lats = [d["lat"] for _, d in nwc]
    lons = [d["lon"] for _, d in nwc]
    degrees = [G.degree(n) for n, _ in nwc]
    max_deg = max(degrees) if degrees else 1
    sizes = [2 + 30 * (d / max_deg) for d in degrees]

    fig, ax = plt.subplots(figsize=(8, 11))
    sc = ax.scatter(lons, lats, c=degrees, s=sizes, cmap="viridis", alpha=0.55, linewidths=0)
    plt.colorbar(sc, ax=ax, label="Degree")
    ax.set_xlabel("קו אורך")
    ax.set_ylabel("קו רוחב")
    ax.set_title(f"מפת תחנות תחבורה ציבורית ישראל\n({len(nwc):,} תחנות פעילות)")
    plt.tight_layout()
    plt.savefig(fig_dir / "network_overview_map.png", dpi=150)
    plt.show()


plot_network_map(G, FIG_DIR)

## גרפים: גדלי רכיבים ופיזור אזורי

In [ ]:
def plot_component_sizes(G, fig_dir):
    sizes = sorted([len(c) for c in nx.connected_components(G)], reverse=True)[:20]
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ["#2563eb" if i == 0 else "#94a3b8" for i in range(len(sizes))]
    ax.bar(range(1, len(sizes)+1), sizes, color=colors)
    ax.set_xlabel("רכיב קשור (#)")
    ax.set_ylabel("מספר תחנות")
    ax.set_title("גדלי רכיבים קשורים (Top 20)")
    ax.set_yscale("log")
    plt.tight_layout()
    plt.savefig(fig_dir / "components_summary_bar.png", dpi=150)
    plt.show()


def plot_region_distribution(G, fig_dir):
    regions = {}
    for n, d in G.nodes(data=True):
        r = d.get("region", "לא ידוע")
        regions[r] = regions.get(r, 0) + 1
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(regions.keys(), regions.values(), color=["#2563eb", "#16a34a", "#dc2626", "#d97706"])
    ax.set_xlabel("אזור")
    ax.set_ylabel("מספר תחנות")
    ax.set_title("פיזור תחנות לפי אזור גיאוגרפי")
    for bar, val in zip(bars, regions.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, f"{val:,}", ha="center", va="bottom")
    plt.tight_layout()
    plt.savefig(fig_dir / "stops_by_region.png", dpi=150)
    plt.show()


plot_component_sizes(G, FIG_DIR)
plot_region_distribution(G, FIG_DIR)